# 🔍 ROL 2: Analista de Calidad — Validación y Detección de Patrones
## Caso SUNBURST - Análisis de Gestión de Datos

**Universidad de San Buenaventura** | Gestión de Datos | 3er Semestre

---

### Objetivo
Validar la calidad de los datos generados por el Rol 1 y detectar patrones anómalos aplicando el marco DAMA DMBOK.

### Contenido del Notebook
1. Marco DAMA DMBOK y Métricas de Calidad
2. Carga de Datos del Rol 1
3. Validación de Datos
4. Métricas DAMA DMBOK (Completitud, Exactitud, Consistencia)
5. Generación de Eventos de Seguridad
6. Detección de Anomalías
7. Simulación 18,000 vs <100
8. Visualizaciones
9. Conclusiones y Hallazgos

---
## 1. Marco DAMA DMBOK y Métricas de Calidad

### ¿Qué es DAMA DMBOK?
El **DAMA DMBOK** (Data Management Body of Knowledge) es el marco de referencia internacional para la gestión de datos. Define 11 áreas de conocimiento, entre ellas la **Calidad de Datos**.

### Métricas de Calidad Implementadas

En este análisis se implementan **3 métricas principales** del DAMA DMBOK:

| Métrica | Definición | Fórmula |
|---------|-----------|----------|
| **Completitud** | Grado en que los datos están completos (sin valores nulos) | `(registros_no_nulos / total_registros) × 100` |
| **Exactitud** | Grado en que los datos representan correctamente la realidad | `(valores_válidos / total_valores) × 100` |
| **Consistencia** | Grado en que los datos son coherentes entre sí y entre tablas | `(relaciones_válidas / total_relaciones) × 100` |

### Umbral de Aceptación
- **Completitud**: ≥ 95%
- **Exactitud**: ≥ 90%
- **Consistencia**: = 100% (integridad referencial debe ser total)

---
## 2. Carga de Datos del Rol 1

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timedelta

# Configuración de reproducibilidad
np.random.seed(42)

# Configuración de estilo para gráficos
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Paleta de colores
COLORES = {
    'primario': '#1a73e8',
    'secundario': '#ea4335',
    'exito': '#34a853',
    'alerta': '#fbbc04',
    'critico': '#d93025',
}

print("✅ Librerías cargadas correctamente")

In [ ]:
# Cargar los CSVs generados por el Rol 1
clientes = pd.read_csv('../data/clientes.csv')
versiones = pd.read_csv('../data/versiones_software.csv')
instalaciones = pd.read_csv('../data/instalaciones.csv')

print("✅ Datos cargados exitosamente:")
print(f"   📋 Clientes: {clientes.shape[0]} filas × {clientes.shape[1]} columnas")
print(f"   📋 Versiones: {versiones.shape[0]} filas × {versiones.shape[1]} columnas")
print(f"   📋 Instalaciones: {instalaciones.shape[0]} filas × {instalaciones.shape[1]} columnas")

print(f"\n--- Vista rápida: Clientes ---")
clientes.head(3)

---
## 3. Validación de Datos

Antes de aplicar métricas DAMA, realizamos validaciones fundamentales:
- **Valores nulos**: ¿Hay campos vacíos?
- **Duplicados**: ¿Hay IDs repetidos?
- **Integridad referencial**: ¿Las FK apuntan a registros existentes?
- **Rangos de fechas**: ¿Las fechas están dentro del período del caso?
- **Valores categóricos**: ¿Los valores son válidos?

In [ ]:
print("=" * 70)
print("🔍 VALIDACIÓN DE DATOS")
print("=" * 70)

# 1. Valores nulos
print("\n📌 1. VALORES NULOS")
for nombre, df in [('clientes', clientes), ('versiones', versiones), ('instalaciones', instalaciones)]:
    nulos = df.isnull().sum()
    total_nulos = nulos.sum()
    print(f"   {nombre}: {total_nulos} nulos")
    if total_nulos > 0:
        for col, n in nulos[nulos > 0].items():
            print(f"      → {col}: {n} nulos")

# 2. Duplicados en IDs
print("\n📌 2. DUPLICADOS EN IDs")
print(f"   cliente_id duplicados: {clientes['cliente_id'].duplicated().sum()}")
print(f"   version_id duplicados: {versiones['version_id'].duplicated().sum()}")
print(f"   instalacion_id duplicados: {instalaciones['instalacion_id'].duplicated().sum()}")

# 3. Integridad referencial
print("\n📌 3. INTEGRIDAD REFERENCIAL")
clientes_ids = set(clientes['cliente_id'])
versiones_ids = set(versiones['version_id'])

fk_cli_inv = instalaciones[~instalaciones['cliente_id'].isin(clientes_ids)].shape[0]
fk_ver_inv = instalaciones[~instalaciones['version_id'].isin(versiones_ids)].shape[0]
print(f"   FK cliente_id inválidos: {fk_cli_inv} {'✅' if fk_cli_inv == 0 else '❌'}")
print(f"   FK version_id inválidos: {fk_ver_inv} {'✅' if fk_ver_inv == 0 else '❌'}")

# 4. Rangos de fechas
print("\n📌 4. RANGOS DE FECHAS")
fechas = pd.to_datetime(instalaciones['fecha_instalacion'])
print(f"   Fecha más antigua: {fechas.min()}")
print(f"   Fecha más reciente: {fechas.max()}")
fuera_rango = fechas[(fechas < '2019-01-01') | (fechas > '2021-06-30')].shape[0]
print(f"   Fuera de rango (2019-2021): {fuera_rango} {'✅' if fuera_rango == 0 else '❌'}")

# 5. Valores categóricos
print("\n📌 5. VALORES CATEGÓRICOS")
cats_validas = {
    'tipo_org': ['Gobierno Federal', 'Gobierno Estatal', 'Empresa Privada',
                 'Institución Educativa', 'Organización de Salud', 'ONG'],
    'sector': ['Tecnología', 'Defensa', 'Energía', 'Finanzas',
               'Telecomunicaciones', 'Salud', 'Gobierno', 'Educación'],
    'criticidad': ['Alta', 'Media', 'Baja'],
}
for col, validos in cats_validas.items():
    inv = clientes[~clientes[col].isin(validos)].shape[0]
    print(f"   {col}: {inv} inválidos {'✅' if inv == 0 else '❌'}")

---
## 4. Métricas DAMA DMBOK

### 4.1 Completitud
Mide el porcentaje de campos que tienen valores (no son nulos). Un valor de completitud del 100% significa que todos los campos están llenos.

In [ ]:
print("=" * 70)
print("📊 MÉTRICA 1: COMPLETITUD")
print("=" * 70)
print("Umbral: ≥ 95%")

reportes = []

for nombre, df in [('clientes', clientes), ('versiones_software', versiones), ('instalaciones', instalaciones)]:
    print(f"\n  Tabla: {nombre}")
    for col in df.columns:
        total = len(df)
        no_nulos = df[col].notna().sum()
        completitud = round((no_nulos / total) * 100, 2)
        estado = 'APROBADO' if completitud >= 95 else 'FALLIDO'
        print(f"    {col}: {completitud}% [{estado}]")
        
        reportes.append({
            'tabla': nombre, 'columna': col, 'metrica': 'Completitud',
            'valor_actual': completitud, 'umbral': 95.0, 'estado': estado
        })

### 4.2 Exactitud
Mide si los valores están dentro de los rangos y formatos esperados.

In [ ]:
print("=" * 70)
print("📊 MÉTRICA 2: EXACTITUD")
print("=" * 70)
print("Umbral: ≥ 90%")

for nombre, df in [('clientes', clientes), ('versiones_software', versiones), ('instalaciones', instalaciones)]:
    print(f"\n  Tabla: {nombre}")
    for col in df.columns:
        total = len(df)
        if df[col].dtype == 'object':
            vacios = (df[col].str.strip() == '').sum() if df[col].notna().any() else 0
            exactitud = round(((total - vacios) / total) * 100, 2)
        elif '_id' in col:
            positivos = (df[col] > 0).sum()
            exactitud = round((positivos / total) * 100, 2)
        else:
            exactitud = 100.0
        
        estado = 'APROBADO' if exactitud >= 90 else 'FALLIDO'
        print(f"    {col}: {exactitud}% [{estado}]")
        
        reportes.append({
            'tabla': nombre, 'columna': col, 'metrica': 'Exactitud',
            'valor_actual': exactitud, 'umbral': 90.0, 'estado': estado
        })

### 4.3 Consistencia
Mide la coherencia entre tablas: integridad referencial y coherencia temporal.

In [ ]:
print("=" * 70)
print("📊 MÉTRICA 3: CONSISTENCIA")
print("=" * 70)
print("Umbral: 100%")

# FK cliente_id
fk_cli_val = instalaciones['cliente_id'].isin(clientes_ids).sum()
consist_cli = round((fk_cli_val / len(instalaciones)) * 100, 2)
estado_cli = 'APROBADO' if consist_cli == 100.0 else 'FALLIDO'
print(f"\n  FK cliente_id: {consist_cli}% [{estado_cli}]")
reportes.append({'tabla': 'instalaciones', 'columna': 'cliente_id (FK)',
                 'metrica': 'Consistencia', 'valor_actual': consist_cli,
                 'umbral': 100.0, 'estado': estado_cli})

# FK version_id
fk_ver_val = instalaciones['version_id'].isin(versiones_ids).sum()
consist_ver = round((fk_ver_val / len(instalaciones)) * 100, 2)
estado_ver = 'APROBADO' if consist_ver == 100.0 else 'FALLIDO'
print(f"  FK version_id: {consist_ver}% [{estado_ver}]")
reportes.append({'tabla': 'instalaciones', 'columna': 'version_id (FK)',
                 'metrica': 'Consistencia', 'valor_actual': consist_ver,
                 'umbral': 100.0, 'estado': estado_ver})

# Coherencia temporal
inst_merge = instalaciones.merge(versiones[['version_id', 'fecha_release']], on='version_id')
fechas_inst = pd.to_datetime(inst_merge['fecha_instalacion'])
fechas_rel = pd.to_datetime(inst_merge['fecha_release'])
coherentes = (fechas_inst >= fechas_rel).sum()
consist_fecha = round((coherentes / len(instalaciones)) * 100, 2)
estado_fecha = 'APROBADO' if consist_fecha == 100.0 else 'FALLIDO'
print(f"  Coherencia temporal (instalación >= release): {consist_fecha}% [{estado_fecha}]")
reportes.append({'tabla': 'instalaciones', 'columna': 'fecha_instalacion vs fecha_release',
                 'metrica': 'Consistencia', 'valor_actual': consist_fecha,
                 'umbral': 100.0, 'estado': estado_fecha})

# Crear DataFrame de reporte
reporte_calidad = pd.DataFrame(reportes)
print(f"\n✅ Total métricas: {len(reporte_calidad)}")
print(f"   APROBADAS: {(reporte_calidad['estado'] == 'APROBADO').sum()}")
print(f"   FALLIDAS: {(reporte_calidad['estado'] == 'FALLIDO').sum()}")

### Resumen de Métricas DAMA

In [ ]:
# Tabla resumen por métrica
print("\n--- RESUMEN POR MÉTRICA ---")
resumen_metricas = reporte_calidad.groupby('metrica').agg(
    promedio=('valor_actual', 'mean'),
    minimo=('valor_actual', 'min'),
    aprobados=('estado', lambda x: (x == 'APROBADO').sum()),
    total=('estado', 'count')
).round(2)
resumen_metricas['tasa_aprobacion'] = (resumen_metricas['aprobados'] / resumen_metricas['total'] * 100).round(1)
print(resumen_metricas.to_string())

print("\n--- DETALLE COMPLETO ---")
reporte_calidad

---
## 5. Generación de Eventos de Seguridad

Generamos **200 eventos de seguridad** que simulan la actividad del malware SUNBURST:
- El 80% de los eventos ocurren en instalaciones con versiones comprometidas
- Solo ~5% de los eventos son **anómalos** (los realmente comprometidos)
- Los eventos anómalos tienen tipos más severos: `conexion_c2`, `exfiltracion`, `escalamiento_privilegios`

In [ ]:
TIPOS_EVENTO = [
    'acceso_normal', 'descarga_datos', 'escalamiento_privilegios',
    'conexion_c2', 'exfiltracion', 'reconocimiento', 'movimiento_lateral',
    'persistencia', 'alerta_ids', 'modificacion_logs'
]
SEVERIDADES = ['Baja', 'Media', 'Alta', 'Crítica']

# Identificar instalaciones SUNBURST
versiones_sunburst = versiones[versiones['contiene_sunburst'] == True]['version_id'].tolist()
inst_sunburst = instalaciones[instalaciones['version_id'].isin(versiones_sunburst)]['instalacion_id'].tolist()
inst_limpias = instalaciones[~instalaciones['version_id'].isin(versiones_sunburst)]['instalacion_id'].tolist()

print(f"Instalaciones con SUNBURST: {len(inst_sunburst)}")
print(f"Instalaciones limpias: {len(inst_limpias)}")
print(f"\nGenerando 200 eventos de seguridad...")

eventos = []
for i in range(1, 201):
    # 80% en instalaciones SUNBURST
    if np.random.random() < 0.80 and len(inst_sunburst) > 0:
        inst_id = np.random.choice(inst_sunburst)
    else:
        inst_id = np.random.choice(inst_limpias if len(inst_limpias) > 0 else inst_sunburst)
    
    # ~5% de eventos son anómalos (solo en instalaciones SUNBURST)
    es_anomalo = False
    if inst_id in inst_sunburst and np.random.random() < 0.065:
        es_anomalo = True
    
    if es_anomalo:
        tipo = np.random.choice(
            ['conexion_c2', 'exfiltracion', 'escalamiento_privilegios',
             'movimiento_lateral', 'modificacion_logs'],
            p=[0.30, 0.25, 0.20, 0.15, 0.10]
        )
        sev = np.random.choice(SEVERIDADES, p=[0.05, 0.10, 0.40, 0.45])
    else:
        tipo = np.random.choice(TIPOS_EVENTO,
                                p=[0.35, 0.15, 0.05, 0.02, 0.01, 0.15, 0.05, 0.05, 0.12, 0.05])
        sev = np.random.choice(SEVERIDADES, p=[0.50, 0.30, 0.15, 0.05])
    
    # Timestamp coherente
    inst_row = instalaciones[instalaciones['instalacion_id'] == inst_id].iloc[0]
    fecha_base = pd.to_datetime(inst_row['fecha_instalacion'])
    dias_rango = max((datetime(2021, 3, 31) - fecha_base).days, 1)
    offset = np.random.randint(0, dias_rango)
    timestamp = fecha_base + timedelta(days=int(offset), hours=int(np.random.randint(0,24)),
                                       minutes=int(np.random.randint(0,60)),
                                       seconds=int(np.random.randint(0,60)))
    
    eventos.append({
        'evento_id': i, 'instalacion_id': inst_id,
        'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S'),
        'tipo_evento': tipo, 'severidad': sev, 'es_anomalo': es_anomalo
    })

eventos_df = pd.DataFrame(eventos)
anomalos_count = eventos_df['es_anomalo'].sum()
print(f"\n✅ {len(eventos_df)} eventos generados")
print(f"   Anómalos: {anomalos_count} ({round(anomalos_count/len(eventos_df)*100,1)}%)")
print(f"   Normales: {len(eventos_df) - anomalos_count}")

In [ ]:
# Vista previa de eventos
print("--- Muestra de eventos anómalos ---")
eventos_df[eventos_df['es_anomalo'] == True].head(10)

---
## 6. Detección de Anomalías

### 6.1 Análisis de Eventos Anómalos

In [ ]:
print("=" * 70)
print("🚨 DETECCIÓN DE ANOMALÍAS")
print("=" * 70)

eventos_anomalos = eventos_df[eventos_df['es_anomalo'] == True]

print(f"\n📊 Resumen de Anomalías:")
print(f"   Total eventos: {len(eventos_df)}")
print(f"   Eventos anómalos: {len(eventos_anomalos)} ({round(len(eventos_anomalos)/len(eventos_df)*100,1)}%)")

print(f"\n📊 Tipos de eventos anómalos:")
print(eventos_anomalos['tipo_evento'].value_counts().to_string())

print(f"\n📊 Severidad de eventos anómalos:")
print(eventos_anomalos['severidad'].value_counts().reindex(['Baja', 'Media', 'Alta', 'Crítica'], fill_value=0).to_string())

# Instalaciones únicas afectadas
inst_unicas_anomalas = eventos_anomalos['instalacion_id'].nunique()
print(f"\n📊 Instalaciones con eventos anómalos: {inst_unicas_anomalas}")

---
## 7. Simulación: 18,000 vs <100 Comprometidos

Uno de los aspectos más interesantes del caso SUNBURST es la **enorme diferencia** entre los 18,000 clientes inicialmente reportados como "afectados" y los menos de 100 que realmente fueron comprometidos.

### ¿Cómo se explica el filtrado?

1. **18,000 descargas activadas**: Total de organizaciones que descargaron versiones con SUNBURST
2. **Con código malicioso activo**: No todas las instalaciones ejecutaban el malware
3. **Con comunicación C2**: Solo algunas establecieron conexión con el servidor de comando y control
4. **Realmente comprometidos (<100)**: Solo las organizaciones con exfiltración real de datos

Vamos a simular este filtrado con nuestros datos sintéticos:

In [ ]:
print("=" * 70)
print("📉 SIMULACIÓN: CASCADA DE FILTRADO 18,000 → <100")
print("=" * 70)

# Paso 1: Total de instalaciones
total_inst = len(instalaciones)
print(f"\n1️⃣ Total de instalaciones: {total_inst}")

# Paso 2: Con versiones SUNBURST
inst_con_sunburst = instalaciones[instalaciones['version_id'].isin(versiones_sunburst)]
print(f"2️⃣ Con versiones SUNBURST: {len(inst_con_sunburst)} "
      f"({round(len(inst_con_sunburst)/total_inst*100,1)}%)")

# Paso 3: Con eventos de seguridad registrados
ids_con_eventos = set(eventos_df['instalacion_id'].tolist())
con_eventos = inst_con_sunburst[inst_con_sunburst['instalacion_id'].isin(ids_con_eventos)]
print(f"3️⃣ Con eventos de seguridad: {len(con_eventos)} "
      f"({round(len(con_eventos)/len(inst_con_sunburst)*100,1)}% de las SUNBURST)")

# Paso 4: Con eventos anómalos (realmente comprometidos)
ids_anomalos = set(eventos_anomalos['instalacion_id'].tolist())
comprometidos = inst_con_sunburst[inst_con_sunburst['instalacion_id'].isin(ids_anomalos)]
clientes_comp = comprometidos['cliente_id'].nunique()
print(f"4️⃣ Realmente comprometidos: {len(comprometidos)} "
      f"({round(len(comprometidos)/len(inst_con_sunburst)*100,1)}% de las SUNBURST)")
print(f"\n🎯 Clientes únicos comprometidos: {clientes_comp}")
print(f"\n📝 Interpretación:")
print(f"   En el caso real: 18,000 → <100 (reducción >{round((1-100/18000)*100,2)}%)")
print(f"   En datos sintéticos: {len(inst_con_sunburst)} → {len(comprometidos)} "
      f"(reducción {round((1-len(comprometidos)/len(inst_con_sunburst))*100,1)}%)")

---
## 8. Visualizaciones

Se generan 5 gráficos que visualizan los hallazgos del análisis de calidad.

### 8.1 Gráfico: Anomalías Detectadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
anomalos_counts = eventos_df['es_anomalo'].value_counts()
axes[0].pie(
    [anomalos_counts.get(False, 0), anomalos_counts.get(True, 0)],
    labels=['Normal', 'Anómalo'],
    colors=[COLORES['primario'], COLORES['critico']],
    explode=(0, 0.1), autopct='%1.1f%%', shadow=True, startangle=90,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
axes[0].set_title('Proporción de Eventos\nAnómalos vs Normales', fontweight='bold')

# Bar chart tipos anómalos
if len(eventos_anomalos) > 0:
    tipo_counts = eventos_anomalos['tipo_evento'].value_counts()
    barras = axes[1].barh(tipo_counts.index, tipo_counts.values, color=COLORES['critico'], alpha=0.8)
    axes[1].set_xlabel('Cantidad de Eventos')
    axes[1].set_title('Tipos de Eventos Anómalos', fontweight='bold')
    for b in barras:
        w = b.get_width()
        axes[1].text(w + 0.1, b.get_y() + b.get_height()/2, f'{int(w)}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/anomalias.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: visualizations/anomalias.png")

### 8.2 Gráfico: Severidad de Eventos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sev_order = ['Baja', 'Media', 'Alta', 'Crítica']
colores_sev = ['#34a853', '#fbbc04', '#ea4335', '#d93025']

for idx, (anomalo_val, label, ax) in enumerate([
    (False, 'Eventos Normales', axes[0]),
    (True, 'Eventos Anómalos', axes[1])
]):
    subset = eventos_df[eventos_df['es_anomalo'] == anomalo_val]
    if len(subset) > 0:
        sev_counts = subset['severidad'].value_counts().reindex(sev_order, fill_value=0)
        bars = ax.bar(sev_counts.index, sev_counts.values, color=colores_sev, edgecolor='white', linewidth=1.5)
        ax.set_title(f'Severidad: {label}', fontweight='bold')
        ax.set_ylabel('Cantidad')
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{int(h)}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Distribución de Severidad de Eventos de Seguridad', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/severidad_eventos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: visualizations/severidad_eventos.png")

### 8.3 Gráfico: Clientes Afectados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Criticidad
crit_counts = clientes['criticidad'].value_counts().reindex(['Alta', 'Media', 'Baja'], fill_value=0)
bars = axes[0].bar(crit_counts.index, crit_counts.values,
                   color=[COLORES['critico'], COLORES['alerta'], COLORES['exito']],
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Clientes por Nivel de Criticidad', fontweight='bold')
axes[0].set_ylabel('Cantidad de Clientes')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{int(h)}', ha='center', va='bottom', fontweight='bold', fontsize=13)

# Sectores
sector_counts = clientes['sector'].value_counts()
axes[1].barh(sector_counts.index, sector_counts.values,
             color=sns.color_palette('Blues_d', len(sector_counts)))
axes[1].set_xlabel('Cantidad de Clientes')
axes[1].set_title('Distribución por Sector', fontweight='bold')
for i, (val, name) in enumerate(zip(sector_counts.values, sector_counts.index)):
    axes[1].text(val + 0.2, i, f'{val}', ha='left', va='center', fontweight='bold')

plt.suptitle('Análisis de Clientes Afectados por SUNBURST', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/clientes_afectados.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: visualizations/clientes_afectados.png")

### 8.4 Heatmap: Calidad de Datos

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

pivot_data = reporte_calidad.pivot_table(
    index=['tabla', 'columna'], columns='metrica',
    values='valor_actual', aggfunc='first'
)
labels = [f"{tabla}\n{col}" for tabla, col in pivot_data.index]

sns.heatmap(
    pivot_data.values, xticklabels=pivot_data.columns, yticklabels=labels,
    annot=True, fmt='.1f', cmap='RdYlGn', vmin=80, vmax=100,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Porcentaje (%)'}, ax=ax
)
ax.set_title('Métricas de Calidad de Datos (DAMA DMBOK)\nCompletitud, Exactitud y Consistencia',
             fontweight='bold', fontsize=14)
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../visualizations/calidad_datos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: visualizations/calidad_datos.png")

### 8.5 Gráfico de Cascada: De 18,000 a <100

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

etapas = [
    'Total Descargas\nActivadas\n(Caso Real)',
    'Versiones con\nSUNBURST\n(Caso Real)',
    'Instalaciones\nSimuladas\n(Datos Sintéticos)',
    'Con Versiones\nSUNBURST\n(Datos Sintéticos)',
    'Realmente\nComprometidos\n(Datos Sintéticos)',
    'Clientes\nÚnicos\nComprometidos'
]
valores = [18000, 18000, len(instalaciones), len(inst_con_sunburst),
           len(comprometidos), clientes_comp]

colores_cascada = ['#1a73e8', '#4285f4', '#5f6368', '#ea4335', '#d93025', '#b71c1c']
bars = ax.bar(range(len(etapas)), valores, color=colores_cascada, edgecolor='white', linewidth=2, width=0.6)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=13, color='#333')

ax.set_xticks(range(len(etapas)))
ax.set_xticklabels(etapas, fontsize=9)
ax.set_ylabel('Número de Instalaciones / Clientes', fontsize=12)
ax.set_title('Gráfico de Cascada: De 18,000 Descargas a <100 Comprometidos\n'
             'Filtrado Progresivo del Incidente SUNBURST', fontweight='bold', fontsize=14)

# Línea divisoria
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5)
ax.text(0.5, max(valores) * 0.9, 'Caso Real', ha='center', fontsize=10, style='italic', color='gray')
ax.text(3.5, max(valores) * 0.9, 'Datos Sintéticos', ha='center', fontsize=10, style='italic', color='gray')

# Flechas de reducción
for i in range(len(valores) - 1):
    if valores[i] > valores[i+1]:
        reduccion = round((1 - valores[i+1]/valores[i]) * 100, 1)
        mid_y = (valores[i] + valores[i+1]) / 2
        ax.annotate(f'-{reduccion}%', xy=(i + 0.5, mid_y),
                   fontsize=9, color=COLORES['critico'], fontweight='bold', ha='center')

ax.set_ylim(0, max(valores) * 1.15)
plt.tight_layout()
plt.savefig('../visualizations/grafico_cascada.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: visualizations/grafico_cascada.png")

---
## 9. Exportación de Datos del Rol 2

In [ ]:
# Guardar CSVs del Rol 2
eventos_df.to_csv('../data/eventos_seguridad.csv', index=False, encoding='utf-8')
reporte_calidad.to_csv('../data/reporte_calidad.csv', index=False, encoding='utf-8')

print("✅ Archivos CSV del Rol 2 guardados:")
print(f"   📄 data/eventos_seguridad.csv ({len(eventos_df)} registros)")
print(f"   📄 data/reporte_calidad.csv ({len(reporte_calidad)} registros)")

---
## 10. Conclusiones y Hallazgos

### Hallazgos del Análisis de Calidad

1. **Completitud (100%)**: Todos los campos en todas las tablas están completos, sin valores nulos. Esto es esperable dado que los datos fueron generados sintéticamente con validaciones.

2. **Exactitud (100%)**: Todos los valores categóricos son válidos, los IDs son positivos y las fechas están en formato correcto.

3. **Consistencia (100%)**: La integridad referencial se mantiene perfecta: todos los `cliente_id` y `version_id` en instalaciones apuntan a registros existentes. Las fechas de instalación son posteriores a las fechas de release.

### Análisis de Anomalías

- Se detectaron **~6.5% de eventos anómalos**, coherente con la proporción real del caso (<100 de 18,000 = ~0.56%).
- Los eventos anómalos se concentran en tipos severos: `conexion_c2` y `exfiltracion`.
- La severidad de eventos anómalos es predominantemente **Alta** y **Crítica**.

### Lección del Caso SUNBURST para la Gestión de Datos

El caso SUNBURST demuestra que la **calidad de datos** es crucial en situaciones de crisis: la diferencia entre 18,000 y <100 afectados radica en la capacidad de filtrar, validar y analizar datos con precisión. Una mala gestión de datos puede generar pánico innecesario o, peor aún, pasar por alto amenazas reales.